# Defence Layers -- DP, Robust Aggregation & Composition (Kaggle)

Runs the defence-layer experiments on NIH ChestX-ray14 and produces three results +
charts + a report:
1. **Differential privacy** -- utility cost across noise levels (privacy-utility trade-off).
2. **Robust aggregation** -- FedAvg vs Krum / trimmed-mean / median under a poisoning attack.
3. **Composition (G1)** -- DP x robust aggregation together under attack.

Uses **6 hospital nodes** (so robust aggregation has an honest majority) with ~2/6
malicious. Smaller/faster than the core run because it does ~9 federated runs, some
with DP. Default ~3 h; use **Save Version -> Save & Run All (Commit)** for an unattended run.

## Before you run (right sidebar)
1. **Accelerator -> GPU**, 2. **Internet -> On**, 3. **+ Add Input**:
   `nih-chest-xrays/data` (label CSVs) and optionally
   `khanfashee/nih-chest-x-ray-14-224x224-resized` (faster images).


## 1. GPU check

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Sidebar -> Accelerator -> GPU, then Run All again.")

## 2. Install deps + clone repo (needs Internet = On)

In [ ]:
!pip install -q "flwr>=1.8,<2.0" "opacus>=1.4" torchmetrics scikit-learn pyyaml typer
import os
REPO_URL = "https://github.com/Pooja-Goel07/Federated-Learning-for-Privacy-Preserving-Medical-Diagnosis.git"
if not os.path.exists("/kaggle/working/fl"):
    !git clone -q $REPO_URL /kaggle/working/fl
%cd /kaggle/working/fl
!git log --oneline -1

## 3. Locate the mounted dataset

Finds the label CSVs and the image folder under `/kaggle/input` (no download). Prints
what is attached so a missing input is obvious.

In [ ]:
import glob, os, shutil
import pandas as pd
NIH = "/kaggle/working/nih"
os.makedirs(NIH, exist_ok=True)

print("Contents of /kaggle/input:")
for d in sorted(glob.glob("/kaggle/input/*")):
    csvs = [os.path.basename(c) for c in glob.glob(d + "/**/*.csv", recursive=True)]
    pngs = sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True))
    print(f"  {d}  | {pngs} png | csv: {csvs[:6]}")

all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)
labels = next((c for c in all_csv if os.path.basename(c).lower() == "data_entry_2017.csv"), None)
if labels is None:
    for c in all_csv:
        try:
            if {"Image Index", "Finding Labels"}.issubset(pd.read_csv(c, nrows=1).columns):
                labels = c; break
        except Exception:
            pass
assert labels, "No labels CSV found. '+ Add Input' -> 'nih-chest-xrays/data'."
shutil.copy(labels, f"{NIH}/Data_Entry_2017.csv")
bb = next((c for c in all_csv if os.path.basename(c).lower() == "bbox_list_2017.csv"), None)
if bb:
    shutil.copy(bb, f"{NIH}/BBox_List_2017.csv")

best, best_n = None, 0
for d in sorted(glob.glob("/kaggle/input/*")):
    n = sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True))
    if n > best_n:
        best, best_n = d, n
IMAGE_ROOT = best
print("labels ->", labels, "| bbox:", bb or "none")
print("IMAGE_ROOT:", IMAGE_ROOT, "| png count:", best_n)
assert best_n > 0, "No PNG images in any mounted input."

## 4. Configure the defence sweep

Loads `configs/kaggle_defences.yaml`. `ROBUST` is the aggregation rule carried into the
composition study. Dial `max_images` / `rounds` / `dp_noise` for runtime vs quality.

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/fl")
from src.experiments.config import load_config
from src.experiments.sweep import default_defence_variants

cfg = load_config("configs/kaggle_defences.yaml")
cfg["data"]["root"] = NIH
cfg["data"]["image_root"] = IMAGE_ROOT
cfg["partition"]["manifest_path"] = f"{NIH}/manifest_def.json"
cfg["data"]["subsample"]["max_images"] = 12000   # raise for stronger numbers
cfg["training"]["rounds"] = 8

ROBUST = "median"          # robust rule used in the composition (G1) study
ATTACK = "model_replacement"
variants = default_defence_variants(attack_type=ATTACK, attack_fraction=0.34,
                                    dp_noise=(1.0, 2.0), robust_rule=ROBUST)
print("nodes:", cfg["partition"]["num_nodes"], "| images:", cfg["data"]["subsample"]["max_images"],
      "| rounds:", cfg["training"]["rounds"], "| runs:", len(variants))
for v in variants:
    print("  -", v["name"], "[", v["group"], "]")

## 5. Run the sweep (the long cell, ~3 h)

Each run prints its own per-round progress. Results checkpoint to
`sweep_results.json` after every run, so a disconnect loses at most one run.

In [ ]:
from src.experiments.sweep import run_sweep
OUT = "/kaggle/working/results/defences"
sweep = run_sweep(cfg, variants, OUT, data_root=NIH)

## 6. Charts + defence report

In [ ]:
from src.evaluation.defence_report import generate_defence_report
generate_defence_report(OUT, robust_rule=ROBUST)

from IPython.display import Image, display, Markdown
for p in ["05_dp_utility_tradeoff", "06_robust_under_attack", "07_composition"]:
    display(Image(f"{OUT}/plots/{p}.png"))
display(Markdown(open(f"{OUT}/DEFENCE_REPORT.md").read()))

## 7. Download the results as a zip

Zips the defence results and shows a direct download link.

In [ ]:
import shutil
from IPython.display import FileLink, display
shutil.make_archive("/kaggle/working/fl_defence_results", "zip", OUT)
print("zipped -> /kaggle/working/fl_defence_results.zip")
display(FileLink("fl_defence_results.zip"))